# Exploring MIKE 21 flexible meshes with `pyramids-mesh`

**DHI MIKE 21 FM** (Flexible Mesh) is the shallow-water engine used for coastal and estuarine
hydrodynamics — tides, storm surge, morphology. Its mesh generator writes a plain-text **`.mesh`**
file: a header line, a block of node coordinates (each carrying a bed-level **z**), and an element
table that may hold **triangles only** or a **mix of triangles and quadrilaterals**.

`pyramids-mesh` registers a `mike21` driver that parses that file into a `MeshDataset` — the same
object every other driver in the package returns, so a real DHI mesh drops straight into the
`pyramids` GIS toolkit. This notebook works through **real vendor `.mesh` files** (DHI and MDAL test
data): we read a coastal bathymetry mesh, map its seabed, and then tour the two header layouts and
the two element topologies the format comes in.

## Setup

The only import this adds is `MeshDataset` — the wrapper a driver hands back. Importing from
`pyramids` also puts its vendored GDAL/PROJ data in place, so nothing here is order-sensitive;
import just the names you use.

In [ ]:
from pyramids.netcdf.ugrid import MeshDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cleopatra.basemap.tiles import add_tiles

## Locate the data

The real `.mesh` corpus lives in `examples/data/real-data/mike21/`, three levels up from this
notebook. We address every file with a `pathlib.Path` — the `pyramids` file APIs take a `Path`
directly, never a `str`. We start with a coastal mesh of Kalundborg (Denmark) in geographic
(long/lat) coordinates.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
tri = DATA / "mike21" / "mike21_eumhdr_tri_longlat_3532n_lf.mesh"
print(tri.resolve())

## Read a triangular mesh

A driver's only job is to parse the file and return a `MeshDataset`. We pass the `Path` and name the
`driver` explicitly. The `repr` reports the mesh kind, its node/face counts, and the data variables
carried alongside the topology — here a single `node_z`.

In [ ]:
ds = MeshDataset.read_file(tri, driver="mike21")
ds

## Let cleopatra pick the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Calling its own
chooser once, right after that first read, selects the inline backend in a notebook (and correctly
stays on `Agg` in a plain script or CI), so every figure below embeds natively.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## Explore the dataset

A `MeshDataset` exposes its size, coordinate reference system, data variables and bounds. Note
`crs`/`epsg` read `None`: this file names its projection as `LONG/LAT` in the header, but the driver
does not resolve that token to an EPSG code — the geometry is valid, it is simply not tagged with a
formal CRS. The bounds confirm the long/lat extent (~10.6–11.1 °E, ~55.4–55.8 °N).

In [ ]:
print('nodes :', ds.n_node)
print('faces :', ds.n_face)
print('crs   :', ds.crs)
print('epsg  :', ds.epsg)
print('vars  :', ds.data_variable_names)
print('bounds:', ds.bounds)

### Per-variable summary

`ds.summary()` returns a pandas `DataFrame` — one row per data variable with count, min, max, mean
and standard deviation. Here `node_z` runs from about **-55.7 m** to **+6.3 m**: negative values are
water depth below the reference level, positive values are land/intertidal above it.

In [ ]:
ds.summary()

## `node_z` is bathymetry — map the seabed

Unlike a synthetic fixture, a real MIKE mesh carries a meaningful **`node_z`**: the bed level sampled
at every node. Because it is a node-located variable, `ds.plot("node_z", ...)` paints it straight
onto the mesh with smooth shading between nodes. The tile guard `if ds.epsg:` stays dark because this
file has no resolved CRS.

In [ ]:
_g = ds.plot("node_z", cmap="viridis", colorbar=True, title="node_z — bed level (m)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The map is a real bathymetric field: a deep channel (dark, strongly negative) threads through the
domain while the shallows and shoreline (bright, near zero or positive) frame it. This is exactly the
information a MIKE 21 FM run needs — the mesh is denser where the bathymetry varies fastest.

## Wireframe — the mesh skeleton

`plot_outline()` draws the bare mesh: every face edge, no data colouring. It is the quickest way to
read the element shapes and, crucially, where the mesh refines — the triangles shrink toward the
features the modeller cared about.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The wireframe shows the hallmark of a flexible mesh: a graded triangulation, fine along the coast
and the channel, coarse in the open, unresolved water. That variable resolution is the whole point of
an unstructured solver.

## GIS for free — faces as vectors

Because a `MeshDataset` is a first-class `pyramids` object, the geospatial toolkit works on it with no
conversion step. `to_geodataframe()` turns every face into a shapely polygon in a GeoPandas
`GeoDataFrame` — ready for a spatial join, an overlay, or export to any OGR vector format.

In [ ]:
gdf = ds.to_geodataframe()
gdf.head()

### Outer boundary — `bounding_polygon()`

`bounding_polygon()` dissolves the faces into a single shapely polygon tracing the domain outline —
the land-and-sea boundary of the model.

In [ ]:
poly = ds.bounding_polygon()
import geopandas as gpd
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="crimson", figsize=(5, 4))
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
plt.gca().set_aspect("equal")

The crimson ring is the model's coastline and open boundary dissolved from thousands of triangle
edges into one polygon — the bridge from a solver mesh to a standard GIS geometry.

## Header-layout tour — two headers, one object

MIKE `.mesh` files come with **two header layouts**. The newer **EUM** header has four fields
(`<eum_item> <eum_unit> <n_nodes> <projection>`); the older **legacy** header has just two
(`<n_nodes> <projection>`). The 2-field variant was once rejected by the driver and now reads. We
read one of each — across both topologies — and tabulate what comes back.

In [ ]:
specs = [
    ("mike21_eumhdr_tri_longlat_3532n_lf.mesh",     "eumhdr (4-field)",   "tri",      "LONG/LAT"),
    ("mike21_eumhdr_quadtri_longlat_12n_lf.mesh",   "eumhdr (4-field)",   "quad+tri", "LONG/LAT"),
    ("mike21_legacyhdr_tri_utm33_2057n_lf.mesh",    "legacyhdr (2-field)","tri",      "UTM-33"),
    ("mike21_legacyhdr_quadtri_utm33_535n_lf.mesh", "legacyhdr (2-field)","quad+tri", "UTM-33"),
]
rows = []
for fn, hdr, topo, proj in specs:
    d = MeshDataset.read_file(DATA / "mike21" / fn, driver="mike21")
    rows.append({
        "file": fn.replace("mike21_", "").replace(".mesh", ""),
        "header": hdr, "topology": topo, "projection": proj,
        "type": type(d).__name__, "n_node": d.n_node, "n_face": d.n_face,
    })
pd.DataFrame(rows)

Every row is the same kind of object — a 2D `MeshDataset` with a `node_z` variable — regardless of
header layout, topology or projection string. The driver absorbs the format differences so downstream
code never has to care which MIKE vintage wrote the file.

## Topology contrast — triangles vs mixed quad+tri

The element table is where tri and quad+tri meshes differ. `face_node_connectivity.data` has shape
`(n_face, k)`, where `k` is the widest element: **3** for a pure-triangle mesh, **4** once any
quadrilateral appears — and triangles in a mixed mesh are then padded with a **`-1`** fill in their
unused fourth slot.

In [ ]:
tri_ds  = MeshDataset.read_file(DATA / "mike21" / "mike21_eumhdr_tri_longlat_3532n_lf.mesh", driver="mike21")
quad_ds = MeshDataset.read_file(DATA / "mike21" / "mike21_eumhdr_quadtri_wktutm17_798n_lf.mesh", driver="mike21")
for name, d in [("pure tri", tri_ds), ("mixed quad+tri", quad_ds)]:
    conn = d.mesh.face_node_connectivity.data
    fill = int((conn == -1).sum())
    print(f"{name:16s} conn shape {str(conn.shape):12s} fill(-1) slots: {fill}")

The pure-triangle mesh is 3 columns wide with no fill; the mixed mesh is 4 columns wide and uses the
`-1` sentinel wherever a triangle sits in a quad-capable table. We can count the two element kinds
directly from that fill pattern.

In [ ]:
conn = quad_ds.mesh.face_node_connectivity.data
nverts = (conn != -1).sum(axis=1)
counts = pd.Series(nverts).value_counts().sort_index()
counts.index = counts.index.map({3: "triangle", 4: "quad"})
counts.plot.bar(title="element types in the mixed mesh", rot=0, color=["#4c72b0", "#dd8452"])
plt.ylabel("count")

The mixed mesh is mostly triangles with a band of quadrilaterals — a common MIKE pattern where
quads line a channel or a structured strip and triangles fill the rest. The driver carries both in a
single padded connectivity table.

## A projection may be named, yet `epsg` is `None`

Even the richest projection encoding does not give you an EPSG. The mixed mesh above stores a **full
WKT** `PROJCS[...]` string in its header, so `ds.crs` is populated — but the driver still does not
resolve it to a numeric code, so `ds.epsg` is `None`. For the named-string files (`LONG/LAT`,
`UTM-33`) even `crs` comes back `None`. That is why every `if ds.epsg:` tile guard in this notebook
stays dark — mapping against a basemap needs a resolved CRS you would have to set yourself.

In [ ]:
w = MeshDataset.read_file(DATA / "mike21" / "mike21_eumhdr_quadtri_wktutm17_798n_lf.mesh", driver="mike21")
print("crs populated? ", w.crs is not None)
print("epsg          :", w.epsg)

## Takeaway

The `mike21` driver reads a real DHI `.mesh` — either header layout, triangles or mixed quad+tri,
any projection encoding — into one uniform `MeshDataset`. Node **bathymetry** arrives as `node_z` and
maps straight away; the mesh then flows into the same vector / boundary / GIS toolkit as every other
format. The one caveat: the projection string is preserved but not resolved to an EPSG, so attach a
CRS yourself before you need basemaps or reprojection.